In [ ]:
import os
from ratelimit import limits, sleep_and_retry
import requests
from urllib3.util import Retry
import sqlite3
import json
from pathlib import Path

In [ ]:
x = %pwd
PROJECT_ROOT = Path(x).resolve().parent
DATA_DIRECTORY = PROJECT_ROOT / 'data'

In [ ]:
conn = sqlite3.connect(DATA_DIRECTORY / 'league_data.db')
cursor = conn.cursor()

In [22]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS match_queue (
        match_id TEXT PRIMARY KEY
        , status TEXT DEFAULT 'pending')
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS matches (
        match_id TEXT PRIMARY KEY,
        champ_1 TEXT, champ_2 TEXT, champ_3 TEXT, champ_4 TEXT, champ_5 TEXT,
        champ_6 TEXT, champ_7 TEXT, champ_8 TEXT, champ_9 TEXT, champ_10 TEXT)
''')

conn.commit()

In [ ]:
EPOCH_TIME_JULY2026 = 1782867600
RANKED_SOLO = 420
ONE_SECOND = 1
TWO_MINUTES = 120
NUM_CHAMPIONS_PER_GAME = 10
NUM_GAMES_PER_PLAYER = 1
CURRENT_PATCH = '16.15.1' # Update this with the current patch version

platforms = ['OC1', 'JP1', 'KR', 'BR1', 'LA1', 'LA2', 'NA1', 'TR1', 'RU', 'EUN1', 'EUW1', 'ME1', 'SG2', 'TW2', 'VN2']
regions = {'OC1':'sea', 'SG2': 'sea', 'TW2': 'sea', 'VN2': 'sea', 'JP1': 'asia', 'KR': 'asia', 'BR1': 'americas', 'LA1': 'americas', 'LA2': 'americas', 'NA1': 'americas', 'TR1': 'europe', 'RU': 'europe', 'EUN1': 'europe', 'EUW1': 'europe', 'ME1': 'europe'}

champion_names_url = 'https://ddragon.leagueoflegends.com/cdn/{version}/data/en_US/champion.json'
master_division_url = 'https://{platform}.api.riotgames.com/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5'
matches_by_player_url = 'https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?startTime={start_time}&queue={queue}&type=ranked&start=0&count={count}'
match_data_from_matchid = 'https://{region}.api.riotgames.com/lol/match/v5/matches/{matchId}'
api_key = os.getenv("RIOT_API_KEY")

headers = {
    'X-Riot-Token': api_key
}

In [24]:
session = requests.Session()
retries = Retry(total=10,
                backoff_factor=2,
                status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', requests.adapters.HTTPAdapter(max_retries=retries))

In [25]:
@sleep_and_retry
@limits(calls=95, period=TWO_MINUTES)
@limits(calls=18, period=ONE_SECOND)
def call_api(url, headers=None):
    response = session.get(url, headers=headers)

    if response.status_code >= 400:
        print(f'Status: {response.status_code} Url: {url}')
        return None
    
    return response

In [27]:
# currently up to br1
for platform in platforms[3:]:
    player_data = call_api(master_division_url.format(platform=platform), headers)

    data = player_data.json()['entries']
    player_id = [player['puuid'] for player in data] # collects all the player puuids from the master division

    for puuid in player_id:
        url = matches_by_player_url.format(region=regions[platform],
                                           puuid=puuid,
                                           start_time=EPOCH_TIME_JULY2026,
                                           queue=RANKED_SOLO,
                                           count=NUM_GAMES_PER_PLAYER)
        response = call_api(url, headers)
        
        if not response:
            continue

        for match in response.json():
            cursor.execute('INSERT OR IGNORE INTO match_queue (match_id) VALUES (?)', (match,))

        conn.commit()
        print(f'Added match {match} to queue')

Added match BR1_3267259108 to queue
Added match BR1_3267246450 to queue
Added match BR1_3267268979 to queue
Added match BR1_3266377150 to queue
Added match BR1_3266328731 to queue
Added match BR1_3266171526 to queue
Added match BR1_3266285917 to queue
Added match BR1_3266338378 to queue
Added match BR1_3267205225 to queue
Added match BR1_3266149519 to queue
Added match BR1_3266268505 to queue


KeyboardInterrupt: 

In [9]:
while True:

    cursor.execute('SELECT match_id FROM match_queue WHERE status = "pending" LIMIT 1')
    row = cursor.fetchone()
    
    if not row:
        break

    match_id = row[0]
    platform = match_id.split('_')[0]
    url = match_data_from_matchid.format(region=regions[platform],
                                         matchId=match_id)
    
    response = call_api(url, headers)

    if not response:
        cursor.execute("UPDATE match_queue SET status = 'failed' WHERE match_id = (?)", (match_id,))
        continue
    
    players = response.json()['info']['participants']
    current_champs = [player['championName'] for player in players]

    cursor.execute('''INSERT OR IGNORE INTO matches (
        match_id, champ_1, champ_2, champ_3, champ_4, champ_5, champ_6, champ_7, champ_8, champ_9, champ_10 )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)''', [match_id] + current_champs)

    cursor.execute('UPDATE match_queue SET status = "processed" WHERE match_id = (?)', (match_id,))
    conn.commit()

cursor.close()
conn.close()

In [ ]:
# get champion names
all_champion_names = list(call_api(champion_names_url.format(version=CURRENT_PATCH)).json()['data'].keys())
with open(DATA_DIRECTORY / 'champ_names.json', 'w') as f:
    json.dump(all_champion_names, f)
session.close()